## Install the Libraries


In [1]:
import torch
import math
import torch.nn as nn 
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional

## Class that represents the parameters of the model

In [2]:
@dataclass
class ModelArg:
    dim : int = 4096
    n_layers :int = 32
    n_heads: int = 32 #The number of heads for the Queries
    n_kv_heads: Optional[int] = None #The number of heads of K and V 
    vocab_size : int = -1 #This will be set when we load Tokenizer
    
    #These 2 parameter indicate the hidden dimension of FeedForward Neural Network
    multiple_of: int = 256
    ffn_dim_multiplier : Optional[float] = None
    norm_eps: float = 1e-5 
    #Parameters for KV Cache
    max_batch_size: int = 32
    max_seq_len: int = 2048
    
    device: str = None

## Main Architecture Class 

- Efficiency: Instead of re-processing the entire sequence every step, KV caching allows the model to only compute attention for the new token against stored KV pairs. This makes inference much faster.

In [ ]:
def precompute_theta_pos_frequencies(head_dim: int, seq_len:int , device:str, theta:float = 10000.0):
    #Since the rotational positional encodings can't be implied for embeddings  having ODD dimensions
    assert head_dim % 2 == 0, "Dimension of the embeddings must be EVEN"
    
    #Calculate the theta parameters according to the formula:- theta_i = 10000^(-2(i-1)/dim) for i in range(1, dim_head/2) 
    #The Shape : (head_dim /2)
    theta_numerator = torch.arange(0, head_dim, 2).float()
    
    theta = 1.0/ (theta ** (theta_numerator/head_dim)).to(device)
    
    #Cosntruct the 'm' parameters which are the index of COSINE & SINE values.SHAPE will be (seq_len)
    m = torch.arange(seq_len, device=device)
    
    #Compute the Outer Product to get frequencies
    freqs = torch.outer(m ,theta ).float() #torch.outer():- Multiplies one  element of first vector with all the elements of second vector consecutively
    
    #We can compute complex number in the polar form c = R * exp(i * m * theta ), where R = 1
    #(Seq_len , Head_dim/2 ) -> (Seq_len , Head_dim/2)
    freq_complex = torch.polar(torch.ones_like(freqs), freqs)
    
    return freq_complex

#Recommended to watch UMAR JAMIL video on Implementation from 42:00 to get the idea behind the rotary positional embeddings function
def apply_rotary_embeddings(x : torch.Tensor, freq_complex:torch.Tensor , device:str):
    #freq_complex will only be for X tensor and all the other freq_complex will not be taken
    
    #(Batch_dim, seq_len, H, head_dim) ------------> (Batch_dim, Seq_len , H, head_dim/2 ) {head_dim/2 because every 2 consecutive pairs are becoming 1 complex number}
    #x.float().reshape(*x.shape[:-1],-1, 2):- This operation is taking 2 consecutive dimensions and grouping them together
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1],-1, 2))
    
    
    #(batch_dim, head_dim/2 ) ------> (1, seq_len, 1 , head_dim/2)
    freq_complex = freq_complex.unsqueeze(0).unsqueeze(2)
    
    #(batch_dim, seq_len, h , head_dim/2)*(1,seq_len, 1, head_dim/2) ----------> (batch_dim, seq_len, h, head_dim/2)
    x_rotated = x_complex * freq_complex
    
    #(batch_dim, seq_len, h , head_dim/2) -------------> (batch_dim, seq_len, h , head_dim/2, 2)
    x_out = torch.view_as_real(x_rotated)
    
    #(batch_dim, seq_len, h, head_dim/2, 2) ------------> (batch_dim, seq_len, h ,head_dim)
    x_out = x_out.reshape(*x.shape)
    
    return x_out.type_as(x).to(device)


def repeat_kv(x:torch.Tensor , n_rep:int)->torch.Tensor:
    batch_size, seq_len , n_kv_heads , head_dim = x.shape
    if n_rep == 1:
        return x
    else:
        return(
        #(batch, seq_len, n_kv_heads, 1,head_dim)
        x[:,:, :, None, :].expand(batch_size, seq_len, n_kv_heads, n_rep, head_dim).reshape(batch_size, seq_len, n_kv_heads*n_rep, head_dim) 
        )

### RMS Normalization Class

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim:int, epsilon:1e-6):
        super().__init__()
        self.epsilon = epsilon   #This is defined to prevent (divide by 0) error
        
        #Gamma Parameter 
        self.weight = nn.Parameter(torch.ones(dim))
        
    def _norm(self, x:torch.Tensor):
        
        #(batch_dim, seq_len, dim)
        return x*torch.sqrt(x.pow(2).mean(-1, keepdim=True)+ self.epsilon)
    
    def forward(self, x: torch.Tensor):
        
        #(dim) * (batch_dim, seq_len, dim) -------> (batch_dim, seq_len, dim)
        return self.weight*self._norm(x.float()).type_as(x)

## Feed Forward Class


- ### FeedForward Function will be SwiGLU based

- ##### FeedForward(x,W,V,W2) = (Swish(xW) @ xV)W2

- Sigmoid Function has a range (0,1) whereas SiLU Function has range (-infinity , +infinity )

In [ ]:
class FeedForward(nn.Module):
    
    def __init__(self, args:ModelArg):
        super().__init__()
        
        hidden_dim = 4*args.dim 
        hidden_dim = int( 2* hidden_dim/3)
        
        if args.ffn_dim_multiplier is not None:
            hidden_dim = int( args.ffn_dim_multiplier * hidden_dim)
            
        #Round the hidden to the nearest just greater multiple of the 'multiple_of' parameter
        hidden = args.multiple_of * ((hidden + args.multiple_of - 1) // args.multiple_of)
        
        self.w1 = nn.Linear(args.dim, hidden_dim , bias= False)
        self.w2 = nn.Linear(hidden_dim, args.dim, bias = False)
        self.w3 = nn.Linear(args.dim, hidden_dim , bias= False)
    
    def forward(self, x:torch.Tensor):
        swish = F.silu(self.w1(x))
        x_V = self.w3(x)
        x = swish * x_V
        x = self.w2(x)
        return x 
        
        
        

## Encoder Architecture Class

In [ ]:
class EncoderBlock(nn.Module):
    
    def __init__(self, args:ModelArg):
        super().__init__()
        
        self.n_heads = args.n_heads
        self.dim = args.dim
        self.head_dim = args.dim // args.n_heads   #This is equal to dimesnion(4096 for llama2) divided by n_heads(32 for llama2)
        
        self.attention = SelfAttention(args)
        self.feed_forward = FeedForward(args)
        
        #Normalization before the activation block
        self.attention_norm = RMSNorm(args.dim, epsilon=args.norm_eps)
        
        #Normalization AFTER the activation block and BEFORE the FeedForward Block
        self.ff_norm = RMSNorm(args.dim, epsilon=args.norm_eps) 
        
    def forward(self, x:torch.Tensor, start_pos:int, freq_complex:torch.Tensor):
        
        h = x + self.attention.forward(self.attention_norm(x), start_pos, freq_complex)  # h is hidden
        out = h + self.feed_forward.forward(self.ff_norm(x))
        return out

## SelfAttention Class

- Watch Umar Jamil Video for more in depth understanding of the SelfAttention Class

In [ ]:
class SelfAttention(nn.Module):
    
    def __init__(self, args:ModelArg):
        super().__init__()
        
        #This is the number of heads for the Key & Values
        self.n_kv_heads = args.n_heads if args.n_kv_heads is None else args.n_kv_heads
        
        #This is the number of heads for the Queries
        self.n_q_heads = args.n_heads
        
        #This is the number of times the heads of Key & Value should be repeated to match the heads of the Queries
        self.n_rep = self.n_q_heads // self.n_kv_heads
        
        #This is the part of the embedding visualized by each head, Each head will watch the full sentence but only a part of the embedding 
        self.head_dim = args.dim // args.n_heads
        
        #Now we have the wq,wk,wv,wo matrices 
        self.w_q = nn.Linear(args.dim, args.n_heads* self.head_dim , bias=False)
        self.w_k = nn.Linear(args.dim, args.n_kv_heads* self.head_dim, bias= False)
        self.w_v = nn.Linear(args.dim, args.n_kv_heads* self.head_dim , bias=False)
        self.w_o = nn.Linear(args.n_heads*self.head_dim , args.dim, bias=False)
        
        self.cache_k = torch.zeros((args.max_batch_size, args.max_seq_len, self.n_kv_heads , self.head_dim))
        self.cache_v = torch.zeros((args.max_batch_size, args.max_seq_len, self.n_kv_heads, self.head_dim))
        
    def forward(self, x:torch.Tensor, start_pos:int , freq_complex:torch.Tensor):
        batch_size, seq_len ,_ = x.shape  #(Batch, 1 , Dim)
        
        #Apply the Wq,Wk & Wv matrices to Query,Key,Value
        #(batch,1,dim) ---------> (batch, 1 , head_q * head_dim) 
        xq = self.w_q(x)
        
        #(batch,1,dim) ----------> (batch,1, head_kv * head_dim)
        xk = self.w_k(x)
        xv = self.w_v(x)
        
        #(batch,1,head_q * head_dim)--------------------> (batch,1 , head_q , head_dim)
        xq = xq.view(batch_size, seq_len, self.n_q_heads, self.head_dim)
        
        #(batch, 1, head_kv*head_dim)-----------------------> (batch , 1 , head_kv , head_dim)
        xk = xk.view(batch_size, seq_len, self.n_kv_heads , self.head_dim)
        xv = xv.view(batch_size, seq_len, self.n_kv_heads , self.head_dim)
        
        #Does not change the shape of the tensors
        xq = apply_rotary_embeddings(xq , freq_complex, device=x.device)
        xk = apply_rotary_embeddings(xk, freq_complex, device=x.device)
        
        #Replace the entry in the cache for this token
        self.cache_v[:batch_size, start_pos:start_pos+seq_len] = xv
        self.cache_k[:batch_size, start_pos:start_pos+seq_len] = xk
        
        #Get all the cached keys and values so far
        # (batch_dim, seq_len_kv , head_kv , head_dim)
        keys = self.cache_k[:batch_size, 0:start_pos+seq_len]
        values = self.cache_v[:batch_size, 0:start_pos + seq_len]
        
        #Repeat the number of heads of Key & Value to reach the number of heads of the queries
        keys = repeat_kv(keys, self.n_rep)
        values = repeat_kv(values, self.n_rep)
        
        #(batch, 1, head_q , head_dim)----------->(batch, head_q , 1, head_dim)
        xq = xq.transpose(1, 2)
        keys = keys.transpose(1,2)
        values = values.transpose(1,2)
        
        #Now do the calculations according to the formula:- activation_score = Q*K^t / (head_dim)^1/2
        
        #(batch, head_q , 1, head_dim) @ (batch, head_q, head_dim, seq_len_kv)---------->(batch, head_q, 1, seq_len_kv)
        score = torch.matmul(xq , keys.transpose(2,3))/ math.sqrt(self.head_dim)
        score = F.softmax(score.float(), dim=-1).type_as(xq)
        
        #(batch, head_q, 1, seq_len_kv) @ (batch, head_q, seq_len_kv, head_dim)---------->(batch, head_q, 1, head_dim)
        output = torch.matmul(score, values)
        
        #(batch, head_q, 1 , head_dim) ------->(batch, 1 , head_q , head_dim)------->(batch, 1 , dim)
        output = (output.transpose(1,2).contiguous().view(batch_size, seq_len , -1))
        return self.w_o(output) 
        
        
        

## Main Transformer Architecture CLass 


In [ ]:
class Transformer(nn.Module):
    def __init__(self, args:ModelArg)->None:
        super().__init__()
        
        assert self.vocab_size != -1, "Error 4203:Vocabulary Size must be set"
        self.args = args
        self.vocab_size = args.vocab_size
        self.n_layers = args.n_layers #This is the number of layers of the whole model architecture{i.e Nx}
        self.token_embeddings = nn.Embedding(self.vocab_size, args.dim)
        
        self.layers = nn.ModuleList()
        for _ in range(args.n_layers):
            self.layers.append(EncoderBlock(args))  #EncoderBlock is not defined yet
        
        self.norm = RMSNorm(args.dim, eps = args.norm_eps)   
        self.output = nn.Linear(args.dim, self.vocab_size, bias=False)
        
        #Function to precompute the values of Rotary Positional Embeddings
        self.freq_complex =  precompute_theta_pos_frequencies(self.args.dim // self.args.n_heads , self.args.max_seq_len*2 , device = self.args.device)   
        
    def forward(self, tokens:torch.Tensor, start_pos:int):
        #The input we get in this forward function will always have Seq_Len of 1 due to KV Cache 
        
        batch_size, seq_len = tokens.shape
        assert seq_len==1, "Only 1 Token at a time can be processed" 
        #This model is only good for INFERENCE and not for TRAINING as training uses Full-Sequencing Process 
        
        h = self.token_embeddings(tokens)
        
        freq_complex = self.freq_complex[start_pos:start_pos+seq_len]
        
        for layer in self.layers:
            h = layer(h, start_pos, freq_complex)
        h = self.norm(h)
        output = self.output(h).float()
        
        return output
        

## Inference Code will be updated in future

In [ ]:
# from typing import Optional
# import time
# from tqdm import tqdm 
# from pathlib import Path
# import json
# from sentencepiece import SentencePieceProcessor



In [ ]:
# class Llama:
    
#     def __init__(self, model:Transformer , tokenizer , model_args:ModelArg):
#         self.model = model
#         self.tokenizer = tokenizer
#         self.args = model_args
        
#     @staticmethod
    # def build(checkpoint_dir:str , tokenizer_path: str, load_model:bool, max_seq_len: int, max_batch_size:int, device:str):
    #     prev_time = time.time()
    #     if load_model:
    #         checkpoints = sorted(Path(checkpoint_dir).glob('*.path'))
    #         assert len(checkpoints) > 0, "No checkpoints file"
    #         chk_path = checkpoints[0]
    #         print(f'Loading Checkpoint {chk_path}')
    #         checkpoint = torch.load(chk_path , map_location="cpu")
    #         print(f'Loaded checkpoint in {(time.time() - prev_time):.2f}s')
    #         prev_time = time.time()
        
    #     with open(Path(checkpoint_dir) / "params.json" , "r") as f:
    #         params = json.loads(f.read())
    #     model_args: ModelArg = ModelArg(
    #         max_seq_len = max_seq_len,
    #         max_batch_size = max_batch_size,
    #         device = device,
    #         **params
    #     )
        
    #     tokenizer = SentencePieceProcessor()
    #     tokenizer.load(tokenizer_path)
    #     model_args.vocab_size = tokenizer.vocab_size()
        
    #     if device == "cuda":
    #         torch.set_default_tensor_type(torch.cuda.HalfTensor)
            